# 03 · Detection with support vector machines

**A "Human-Like" Stroop Effect No Longer Implies a Human Participant**

Reads `data/processed/trials_all.csv` and `respondent_qc.csv` (from `01_data_processing.Rmd`). Run from `analysis/`.
All output goes to `outputs/tables/` and `outputs/figures/`.

| § | Manuscript | Content | Output |
|---|---|---|---|
| 1-3 | none | Setup, helpers, data | none |
| 4 | Method, *Detection With SVMs › Features* | 8 features per respondent | `respondent_features.csv` |
| 5 | Method, *Models*; *Classifier Evaluation* | SVMs, 25/26 splits, measures | none |
| 6 | Results, *Performance on Lab and AEA Data* | 5,000 splits; **Table 2**, **Figure 3** | `mc_splits_all.csv`, `mc_summary.csv`, `fig3_counts.csv`, `fig3.pdf` |
| 7 | Results, *Application to the Online Sample*; **Table 3** | Refit on all lab and AEA respondents, applied to the online sample | `online_*.csv` |
| 8 | Results, *Positive Predictive Value*; **Table 4** | PPV at assumed contamination rates | `ppv_by_contamination.csv` |
| 9 | Supplement, **Tables S1 and S2** | Robustness to $\nu$ and to the evaluation scheme | `robustness_nu.csv`, `robustness_cv.csv` |
| 10 | Supplement, **Table S3** | Feature values and AUC per feature | `feature_auc.csv`, `feature_descriptives.csv` |

Table S4 is produced by `06_threshold_scenarios.py`, Table S5 by `05_agea_cross_agent.py`. Labels: `+1` = lab (human), `-1` = AEA; AEA is the positive class.
The online sample is never used for training.

## 1 · Setup

In [ ]:
import platform, warnings
from math import sqrt
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import scipy
import sklearn
from scipy import stats
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.svm import OneClassSVM, SVC

warnings.filterwarnings("ignore")

ROOT    = Path.cwd().parent                 # the notebook runs in analysis/
PROC    = ROOT / "data" / "processed"
OUT_TAB = ROOT / "outputs" / "tables"
OUT_FIG = ROOT / "outputs" / "figures"
for d in (OUT_TAB, OUT_FIG):
    d.mkdir(parents=True, exist_ok=True)

# Fixed choices stated in the Method
NU_OCSVM     = 0.08          # one-class SVM: nu
N_SPLITS     = 5000          # Monte Carlo cross-validation; split i uses seed SEED0 + i
SEED0        = 1000
SVM_SEED     = 42            # random_state of the supervised SVM
N_PER_CLASS  = 26            # test respondents per sample and split (25 train, 26 test)
PI_SCENARIOS = (0.05, 0.10, 0.20)   # assumed contamination rates (Table 4)

SAMPLES = ["Lab", "Online", "AEA"]
MODELS  = {"oc": "One-class SVM", "sv": "Supervised SVM"}

mpl.rcParams.update({
    "font.family": "sans-serif", "font.sans-serif": ["Segoe UI", "Arial", "DejaVu Sans"],
    "font.size": 9, "axes.spines.top": False, "axes.spines.right": False,
    "savefig.dpi": 300, "savefig.bbox": "tight", "pdf.fonttype": 42, "figure.dpi": 110,
})
FLAGGED, PASSED = "#e34948", "#2166ac"            # red = flagged as AEA, blue = classified as human
INK, INK2, SURFACE = "#0b0b0b", "#52514e", "#ffffff"

## 2 · Helpers

In [ ]:
def nlz(x, d=2):
    """APA: no leading zero for values bounded by 1."""
    s = f"{x:.{d}f}"
    return s.replace("0.", ".", 1) if s.startswith(("0.", "-0.")) else s

def wilson_ci(x, n, z=1.96):
    p = x / n
    denom = 1 + z**2 / n
    centre = (p + z**2 / (2 * n)) / denom
    half = z * sqrt(p * (1 - p) / n + z**2 / (4 * n**2)) / denom
    return max(0.0, centre - half), min(1.0, centre + half)

def save_csv(df, name):
    df.to_csv(OUT_TAB / f"{name}.csv", index=False)

## 3 · Data

In [ ]:
trials = pd.read_csv(PROC / "trials_all.csv")
trials["ID"] = trials["ID"].astype(str)
qc = pd.read_csv(PROC / "respondent_qc.csv")
qc["ID"] = qc["ID"].astype(str)
assert trials["ID"].nunique() == 153 and (trials.groupby("ID").size() == 288).all()
trials.groupby("sample").agg(respondents=("ID", "nunique"), trials=("ID", "size"), misses=("missed", "sum")).reindex(SAMPLES)

## 4 · Features (Method, *Features*)

Six features for any response time task: mean, standard deviation and skewness of the response times of all trials
with a response; task accuracy over all 288 trials (a miss counts as incorrect); response time drift (least squares
slope of response time on trial number); lag-one autocorrelation of successive responses. Two Stroop features, on mean
correct response times: interference (incongruent − neutral) and facilitation (neutral − congruent). The one online
respondent without a correct incongruent response has an undefined interference and cannot be classified.

In [ ]:
GENERIC_FEATURES = ["rt_mean", "rt_sd", "rt_skew", "acc_mean", "rt_drift", "rt_autocorr"]
STROOP_FEATURES  = ["interference", "facilitation"]
FEATURES = GENERIC_FEATURES + STROOP_FEATURES
FEATURE_LABELS = {"facilitation": "Facilitation (ms)", "rt_skew": "RT skewness", "rt_autocorr": "RT autocorrelation (lag 1)",
                  "rt_mean": "Mean RT (ms)", "rt_sd": "SD of RT (ms)", "rt_drift": "RT drift (ms per trial)",
                  "acc_mean": "Task accuracy", "interference": "Interference (ms)"}   # as in Table S3


def engineer_features(df):
    """One row of the 8 features per respondent."""
    records = []
    for pid, p in df.groupby("ID"):
        p = p.sort_values("trial_number")
        resp = p[~p["missed"].astype(bool)]
        rt = resp["rt"].to_numpy(float)
        if len(rt) < 10:
            continue
        ac = pd.Series(rt).autocorr(lag=1)
        c = resp[resp["correct"] == 1].groupby("congruency")["rt"].mean()
        records.append({"ID": pid, "sample": p["sample"].iloc[0],
                        "rt_mean": rt.mean(), "rt_sd": rt.std(), "rt_skew": float(pd.Series(rt).skew()),
                        "acc_mean": float(np.nanmean(p["correct"].to_numpy(float))),
                        "rt_drift": float(stats.linregress(resp["trial_number"].to_numpy(float), rt).slope),
                        "rt_autocorr": 0.0 if pd.isna(ac) else float(ac),
                        "interference": c.get("incongruent", np.nan) - c.get("neutral", np.nan),
                        "facilitation": c.get("neutral", np.nan) - c.get("congruent", np.nan)})
    return pd.DataFrame(records)


features = engineer_features(trials)
save_csv(features, "respondent_features")
undefined = features.loc[features[FEATURES].isna().any(axis=1), ["ID", "sample"]]
print(features.shape, "| respondents with an undefined feature:", undefined.to_dict("records"))
features.groupby("sample")[FEATURES].mean().T.reindex(columns=SAMPLES).round(3)

## 5 · Models and evaluation (Method, *Models*; *Classifier Evaluation*)

One-class SVM: RBF kernel, $\nu$ = .08, trained on lab respondents only, standardized with the lab training data only.
Supervised SVM: RBF kernel, C = 1, balanced class weights (equal at 25/25), standardized with all its training data.
Each split: 25 lab and 25 AEA respondents for training, the other 26 and 26 for testing.

In [ ]:
sample_of = features.set_index("ID")["sample"]
LAB_IDS = np.array(sorted(sample_of.index[sample_of == "Lab"]))
AEA_IDS = np.array(sorted(sample_of.index[sample_of == "AEA"]))
label_map = {i: (-1 if s == "AEA" else 1) for i, s in sample_of.items()}

def make_split(seed):
    rng = np.random.default_rng(seed)
    lab = rng.choice(LAB_IDS, len(LAB_IDS), replace=False)   # a permutation of the 51 respondents
    aea = rng.choice(AEA_IDS, len(AEA_IDS), replace=False)
    lab_tr = rng.choice(lab, len(lab) // 2, replace=False)   # 25 training respondents per sample
    aea_tr = rng.choice(aea, len(aea) // 2, replace=False)
    train = np.concatenate([lab_tr, aea_tr])
    test = np.concatenate([np.setdiff1d(lab, lab_tr), np.setdiff1d(aea, aea_tr)])  # 26 per sample
    return train, test

def prepare(df, cols, scaler=None):
    keep = df.dropna(subset=cols).reset_index(drop=True)
    y = keep["ID"].map(label_map).to_numpy()
    scaler = scaler or StandardScaler().fit(keep[cols])
    return scaler.transform(keep[cols]), y, scaler, keep

class OneClassLabOnly:
    """One-class SVM with its own scaler, fitted on lab training rows only; takes unscaled features."""
    def __init__(self, scaler, model):
        self.scaler, self.model = scaler, model
    def predict(self, X_raw):
        return self.model.predict(self.scaler.transform(X_raw))
    def decision_function(self, X_raw):
        return self.model.decision_function(self.scaler.transform(X_raw))

def fit_oneclass(X_lab_raw, nu=NU_OCSVM):
    scaler = StandardScaler().fit(X_lab_raw)
    model = OneClassSVM(kernel="rbf", nu=nu, gamma="scale").fit(scaler.transform(X_lab_raw))
    return OneClassLabOnly(scaler, model)

def fit_models(X, y, raw_train):
    """X: training rows scaled with all training rows (supervised SVM); raw_train: the same rows unscaled (one-class SVM)."""
    oc = fit_oneclass(raw_train[y == 1])
    sv = SVC(kernel="rbf", class_weight="balanced", random_state=SVM_SEED).fit(X, y)
    return {"oc": oc, "sv": sv}

def predict_model(m, mdl, Xte_scaled, Rte_raw):
    return mdl.predict(Xte_scaled) if m == "sv" else mdl.predict(Rte_raw)

def split_metrics(y_true, pred):
    """Classifier accuracy, sensitivity (AEA flagged) and specificity (lab not flagged); AEA = -1 is positive."""
    tp = int(((y_true == -1) & (pred == -1)).sum()); fn = int(((y_true == -1) & (pred == 1)).sum())
    fp = int(((y_true == 1) & (pred == -1)).sum());  tn = int(((y_true == 1) & (pred == 1)).sum())
    return {"TP": tp, "FN": fn, "FP": fp, "TN": tn, "Accuracy": (tp + tn) / (tp + fn + fp + tn),
            "Sensitivity": tp / (tp + fn), "Specificity": tn / (tn + fp)}

MAIN = f"All {len(FEATURES)} features"

def run_splits(feat, n_splits):
    rows = []
    for i in range(n_splits):
        tr, te = make_split(SEED0 + i)
        Xtr, ytr, sc, ktr = prepare(feat[feat["ID"].isin(tr)], FEATURES)
        Xte, yte, _, kte = prepare(feat[feat["ID"].isin(te)], FEATURES, sc)
        mods = fit_models(Xtr, ytr, ktr[FEATURES].to_numpy())
        for m, mdl in mods.items():
            rows.append({"split": i, "features": MAIN, "model": MODELS[m],
                         **split_metrics(yte, predict_model(m, mdl, Xte, kte[FEATURES].to_numpy()))})
    return pd.DataFrame(rows)

def summarise_splits(df):
    out = (df.groupby(["features", "model"])[["Accuracy", "Sensitivity", "Specificity"]]
           .agg([("median", "median"), ("p2.5", lambda x: x.quantile(.025)), ("p97.5", lambda x: x.quantile(.975))]))
    out.columns = [f"{a}_{b}" for a, b in out.columns]
    return out.reset_index()

## 6 · Table 2 and Figure 3 (Results, *Performance on Lab and AEA Data*)

5,000 splits (seeds 1000 to 5999), 2 SVMs per split; takes a few minutes.

In [ ]:
splits = run_splits(features, N_SPLITS)
assert splits["split"].nunique() == N_SPLITS
assert ((splits["TP"] + splits["FN"]) == N_PER_CLASS).all() and ((splits["TN"] + splits["FP"]) == N_PER_CLASS).all()
save_csv(splits, "mc_splits_all")
summary = summarise_splits(splits)
save_csv(summary, "mc_summary")
summary.round(3)

In [ ]:
cols = [f"{k}_{q}" for k in ("Accuracy", "Sensitivity", "Specificity") for q in ("median", "p2.5", "p97.5")]
summary.set_index("model")[cols].round(2)   # Table 2: medians [2.5th, 97.5th percentiles]

**Figure 3**: one square per test respondent in an illustrative set of 26 lab and 26 AEA respondents. Counts are the
median sensitivity and specificity × 26, rounded (figure note: 48 of 52 correct for the supervised SVM).

In [ ]:
COLS, SQ, GAP = 13, 1.0, 0.22
W = COLS * (SQ + GAP)
MODEL_NOTE = {"One-class SVM": "trained on lab respondents only", "Supervised SVM": "trained on lab and AEA respondents"}

def block(ax, x0, y0, n_flagged, error_is_flagged):
    for k in range(N_PER_CLASS):
        r, c = divmod(k, COLS)
        flagged = k < n_flagged
        error = flagged if error_is_flagged else not flagged
        ax.add_patch(Rectangle((x0 + c * (SQ + GAP), y0 - r * (SQ + GAP)), SQ, SQ,
                               facecolor=FLAGGED if flagged else PASSED,
                               edgecolor=INK if error else SURFACE, linewidth=1.3 if error else 0.8))

def icon_panel(ax, mdl):
    r = summary[summary.model == mdl].iloc[0]
    caught = int(round(r.Sensitivity_median * N_PER_CLASS))
    passed = int(round(r.Specificity_median * N_PER_CLASS))
    wrong = N_PER_CLASS - passed
    ax.text(0, 3.55, "26 AEA respondents", fontsize=8, color=INK2, va="bottom")
    block(ax, 0, 2.2, caught, error_is_flagged=False)
    ax.text(W + .4, 2.35, f"{caught} flagged /\n{N_PER_CLASS - caught} classified\nas human", fontsize=8, va="center")
    ax.text(0, -.45, "26 lab respondents", fontsize=8, color=INK2, va="bottom")
    block(ax, 0, -1.8, wrong, error_is_flagged=True)
    ax.text(W + .4, -1.65, f"{wrong} wrongly\nflagged", fontsize=8, va="center")
    ax.text(0, -3.3, f"Median accuracy {nlz(r.Accuracy_median)}", fontsize=8, color=INK2, va="top")
    ax.set_xlim(-.2, W + 5.6); ax.set_ylim(-4.1, 6.6)
    ax.set_aspect("equal"); ax.axis("off")
    ax.text(0, 6.55, mdl, fontsize=9.5, va="top")
    ax.text(0, 5.55, MODEL_NOTE[mdl], fontsize=8, color=INK2, va="top", style="italic")
    return {"model": mdl, "caught": caught, "missed": N_PER_CLASS - caught, "passed": passed, "wrongly_flagged": wrong}

fig, axs = plt.subplots(1, 2, figsize=(7.2, 3.0))
fig3_counts = pd.DataFrame([icon_panel(ax, mdl) for ax, mdl in zip(axs, MODELS.values())])
fig.legend(handles=[Rectangle((0, 0), 1, 1, facecolor=FLAGGED, edgecolor=SURFACE, label="flagged as AEA"),
                    Rectangle((0, 0), 1, 1, facecolor=PASSED, edgecolor=SURFACE, label="classified as human"),
                    Rectangle((0, 0), 1, 1, facecolor=SURFACE, edgecolor=INK, linewidth=1.3, label="misclassified (outlined)")],
           loc="upper center", ncol=3, frameon=False, bbox_to_anchor=(0.5, 1.02))
fig.tight_layout(w_pad=1.5)
fig.savefig(OUT_FIG / "fig3.pdf"); fig.savefig(OUT_FIG / "fig3.png")
save_csv(fig3_counts, "fig3_counts")
fig3_counts

## 7 · Table 3 (Results, *Application to the Online Sample*)

Both SVMs are refit on all lab and AEA respondents (the one-class SVM on the 51 lab respondents) and applied to the 50
classifiable online respondents. Proportions with 95% Wilson score intervals. Also: respondents flagged by both SVMs,
quality checks of the classified respondents, and the false alarms expected if all were human.

In [ ]:
online = features[features["sample"] == "Online"]
labeled = features[features["sample"].isin(["Lab", "AEA"])]

Xtr, ytr, sc, ktr = prepare(labeled, FEATURES)
Xo, _, _, keep = prepare(online, FEATURES, sc)
mods = fit_models(Xtr, ytr, ktr[FEATURES].to_numpy())
scores = pd.DataFrame({"ID": keep["ID"], "oc": -mods["oc"].decision_function(keep[FEATURES].to_numpy()),
                       "sv": -mods["sv"].decision_function(Xo)})          # score > 0 = flagged
scores["oc_flag"], scores["sv_flag"] = (scores["oc"] > 0).astype(int), (scores["sv"] > 0).astype(int)
save_csv(scores, "online_scores")

app_rows = []
for m in MODELS:
    k, n = int(scores[f"{m}_flag"].sum()), len(scores)
    lo, hi = wilson_ci(k, n)
    app_rows.append({"model": MODELS[m], "n": n, "flagged": k, "proportion": k / n, "lo": lo, "hi": hi})
app = pd.DataFrame(app_rows)
save_csv(app, "online_application")

N_SCORED = len(scores)
agreement = pd.DataFrame([{"classified": N_SCORED, "flagged_by_both": int((scores["oc_flag"] & scores["sv_flag"]).sum()),
                           "classified_failing_a_quality_check": int(qc[qc["ID"].isin(scores["ID"])]["flag_any"].sum())}])
save_csv(agreement, "online_agreement")

display(app.round(3))   # Table 3
agreement

*"If all 50 online respondents were human, the SVM's false-alarm rate for lab data would produce about 21 false alarms
... (leave-one-out, .41) and about 31 ... (main splits, .62)"*; supervised SVM: .10 and .12, about five to six. The
leave-one-out rates come from §9.

In [ ]:
med = summary.set_index("model")
expected = pd.DataFrame({"model": list(MODELS.values()),
                         "false_alarm_rate_main_splits": [1 - med.loc[m, "Specificity_median"] for m in MODELS.values()],
                         f"expected_false_alarms_in_{N_SCORED}": [N_SCORED * (1 - med.loc[m, "Specificity_median"]) for m in MODELS.values()],
                         "observed_flags": [int(scores["oc_flag"].sum()), int(scores["sv_flag"].sum())]})
save_csv(expected, "online_expected_false_alarms")
expected.round(3)

## 8 · Table 4 (Results, *Positive Predictive Value*)

PPV = sπ / [sπ + (1 − c)(1 − π)], with the unrounded median sensitivity s and specificity c (Table 2) and the assumed
contamination rate π. The note gives the 2.5th percentile of the PPV across the 5,000 splits.

In [ ]:
def ppv_of(s, c, pi):
    return s * pi / (s * pi + (1 - c) * (1 - pi))

ppv_rows = []
for mdl in MODELS.values():
    s_, c_ = float(med.loc[mdl, "Sensitivity_median"]), float(med.loc[mdl, "Specificity_median"])
    g = splits[splits.model == mdl]
    for pi in PI_SCENARIOS:
        with np.errstate(divide="ignore", invalid="ignore"):
            per_split = ppv_of(g["Sensitivity"].to_numpy(), g["Specificity"].to_numpy(), pi)   # undefined if nothing flagged
        ppv_rows.append({"model": mdl, "sensitivity": s_, "specificity": c_, "assumed_contamination": pi,
                         "ppv": ppv_of(s_, c_, pi), "ppv_splits_p2.5": np.nanpercentile(per_split, 2.5)})
ppv = pd.DataFrame(ppv_rows)
save_csv(ppv, "ppv_by_contamination")

ppv.round(3)

## 9 · Tables S1 and S2 (Supplement, *Robustness of Analysis Choices*)

S1: the one-class SVM for $\nu$ from .01 to .50, 1,000 splits (seeds 1000 to 1999). S2: medians and means of the
5,000 splits, repeated stratified 10-fold cross-validation (100 repeats, seed 1; median across repeats) and
leave-one-out cross-validation (Method, *Classifier Evaluation*: "a somewhat higher classifier accuracy for the
one-class SVM (.67)").

In [ ]:
lb = features[features["sample"].isin(["Lab", "AEA"])].dropna(subset=FEATURES).reset_index(drop=True)
y_lb = lb["ID"].map(label_map).to_numpy()

def cv_predict(train_idx, test_idx, nu=NU_OCSVM):
    Xtr, ytr, sc, ktr = prepare(lb.iloc[train_idx], FEATURES)
    Xte, _, _, kte = prepare(lb.iloc[test_idx], FEATURES, sc)
    oc = fit_oneclass(ktr[FEATURES].to_numpy()[ytr == 1], nu)
    sv = SVC(kernel="rbf", class_weight="balanced", random_state=SVM_SEED).fit(Xtr, ytr)
    return oc.predict(kte[FEATURES].to_numpy()), sv.predict(Xte)

def three(y, p):
    m = split_metrics(y, p)
    return m["Accuracy"], m["Sensitivity"], m["Specificity"]

cv_rows = [{"scheme": "Monte Carlo, median", "model": m, "Accuracy": med.loc[m, "Accuracy_median"],
            "Sensitivity": med.loc[m, "Sensitivity_median"], "Specificity": med.loc[m, "Specificity_median"]}
           for m in MODELS.values()]
means = splits.groupby("model")[["Accuracy", "Sensitivity", "Specificity"]].mean()
cv_rows += [{"scheme": "Monte Carlo, mean", "model": m, **means.loc[m].to_dict()} for m in MODELS.values()]

rskf = RepeatedStratifiedKFold(n_splits=10, n_repeats=100, random_state=1)
po, ps, rep_res = np.zeros(len(y_lb)), np.zeros(len(y_lb)), []
for k, (tr, te) in enumerate(rskf.split(lb, y_lb)):
    a, b = cv_predict(tr, te)
    po[te], ps[te] = a, b
    if k % 10 == 9:                                  # one full repeat finished
        rep_res.append({"One-class SVM": three(y_lb, po), "Supervised SVM": three(y_lb, ps)})
for mdl in MODELS.values():
    m_ = np.median(np.array([r[mdl] for r in rep_res]), axis=0)
    cv_rows.append({"scheme": "Repeated 10-fold", "model": mdl, "Accuracy": m_[0], "Sensitivity": m_[1], "Specificity": m_[2]})

po, ps = np.zeros(len(y_lb)), np.zeros(len(y_lb))
for i in range(len(y_lb)):
    a, b = cv_predict(np.setdiff1d(np.arange(len(y_lb)), [i]), np.array([i]))
    po[i], ps[i] = a[0], b[0]
for mdl, p in (("One-class SVM", po), ("Supervised SVM", ps)):
    acc, se, sp = three(y_lb, p)
    cv_rows.append({"scheme": "Leave-one-out", "model": mdl, "Accuracy": acc, "Sensitivity": se, "Specificity": sp})
robust_cv = pd.DataFrame(cv_rows)
save_csv(robust_cv, "robustness_cv")

NU_GRID = (0.01, 0.05, 0.08, 0.10, 0.20, 0.30, 0.50)
nu_rows = []
for nu in NU_GRID:
    res = []
    for i in range(1000):
        tr, te = make_split(SEED0 + i)
        Xtr, ytr, sc, ktr = prepare(features[features["ID"].isin(tr)], FEATURES)
        Xte, yte, _, kte = prepare(features[features["ID"].isin(te)], FEATURES, sc)
        oc = fit_oneclass(ktr[FEATURES].to_numpy()[ytr == 1], nu)
        res.append(three(yte, oc.predict(kte[FEATURES].to_numpy())))
    m_ = np.median(np.array(res), axis=0)
    nu_rows.append({"nu": nu, "Accuracy": m_[0], "Sensitivity": m_[1], "Specificity": m_[2]})
robust_nu = pd.DataFrame(nu_rows)
save_csv(robust_nu, "robustness_nu")

display(robust_nu.round(3))   # Table S1
robust_cv.round(3)   # Table S2

## 10 · Table S3 (Supplement, *Separation by Individual Features*; Results, *Performance on Lab and AEA Data*)

Mean (SD) per sample and AUC per feature across all 51 lab and 51 AEA respondents, without a classifier, in the
direction that separates the samples better: max(AUC, 1 − AUC).

In [ ]:
lab_aea = features[features["sample"].isin(["Lab", "AEA"])]
auc_rows = []
for f in FEATURES:
    d = lab_aea.dropna(subset=[f])
    a = roc_auc_score((d["sample"] == "AEA").astype(int), d[f])
    auc_rows.append({"feature": f, "label": FEATURE_LABELS[f], "auc": max(a, 1 - a), "stroop_feature": f in STROOP_FEATURES})
auc = pd.DataFrame(auc_rows).sort_values("auc", ascending=False).reset_index(drop=True)
save_csv(auc, "feature_auc")

desc = (features.melt(id_vars=["sample"], value_vars=FEATURES, var_name="feature")
        .groupby(["feature", "sample"])["value"].agg(n="count", mean="mean", sd="std", min="min", max="max").reset_index())
desc["sample"] = pd.Categorical(desc["sample"], ["Lab", "AEA", "Online"])
desc = desc.sort_values(["feature", "sample"]).reset_index(drop=True)
save_csv(desc, "feature_descriptives")

auc.round(2)   # Table S3 (AUC column); means and SDs are in feature_descriptives.csv

In [ ]:
print("python", platform.python_version(), "| numpy", np.__version__, "| pandas", pd.__version__,
      "| scipy", scipy.__version__, "| scikit-learn", sklearn.__version__, "| matplotlib", mpl.__version__)
print(f"seeds: splits {SEED0}-{SEED0 + N_SPLITS - 1} | supervised SVM random_state {SVM_SEED} | nu = {NU_OCSVM}")